In [0]:
# -----------------------------
# STEP 1: CREATE SKEWED DATA
# -----------------------------

# Large skewed orders data
orders_data = [(1, i) for i in range(1000)] + [(2, 2001), (3, 3001)]
orders = spark.createDataFrame(orders_data, ["customer_id", "order_id"])

# Small customers table
customers_data = [(1, "A"), (2, "B"), (3, "C")]
customers = spark.createDataFrame(customers_data, ["customer_id", "name"])

print("Orders count:", orders.count())
orders.groupBy("customer_id").count().show()

In [0]:
customers.display()

In [0]:
# -----------------------------
# STEP 2: JOIN WITHOUT SALTING (SKEW PROBLEM)
# -----------------------------

print("Before Salting Join")
joined_before = orders.join(customers, "customer_id")

joined_before.groupBy("customer_id").count().show()

In [0]:

from pyspark.sql.functions import *
# -----------------------------
# STEP 3: APPLY SALTING (ONLY FOR SKEWED KEY = 1)
# -----------------------------

orders_salted = orders.withColumn(
    "salt",
    when(col("customer_id") == 1, floor(rand() * 5)).otherwise(0)
)

# Expand customers table
customers_expanded = customers.withColumn(
    "salt",
    when(col("customer_id") == 1,
         explode(array([lit(i) for i in range(5)]))
    ).otherwise(lit(0))
)

In [0]:
# -----------------------------
# STEP 4: JOIN AFTER SALTING
# -----------------------------

print("After Salting Join")
joined_after = orders_salted.join(
    customers_expanded,
    ["customer_id", "salt"]
)

joined_after.groupBy("customer_id", "salt").count().show()
